# Homework 3  
Sarayu Rao

##1. Building a model

Main Question: What are the best predictors of a driver's finishing position for a given race? 

Data Preparation

In [0]:
#Pyspark Imports
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min, sum, when, regexp_extract
import pyspark.sql.functions as F
import pandas as pd

#ML imports
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV


In [0]:
#Load pitstop dataset
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

#Case necessary columns to integers
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

#Get each driver's average pitstop time for each race
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")


In [0]:
#Load results dataset
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

#Join average pitstop time for each driver's race and rename column
df_race_results = df_results.join(df_avg_pit, on = ["raceId","driverId"])
df_race_results = df_race_results.withColumnRenamed("avg(milliseconds)", "avgPitstop")
display(df_race_results)


In [0]:
#Keep and cast necessary columns to integers for our model
df_race_results = df_race_results.select(["raceId","driverId","resultId","positionOrder","laps","fastestLap","fastestLapTime","fastestLapSpeed","avgPitstop","grid", "rank","fastestLapTime"])

df_race_results = df_race_results.filter((df_race_results["fastestLap"] != "\\N") & (df_race_results["fastestLapTime"] != "\\N") & (df_race_results["fastestLapSpeed"] != "\\N"))
df_race_results = df_race_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                      "laps": col("laps").cast("int"),
                                      "fastestLap": col("fastestLap").cast("int"),
                                      "fastestLapSpeed": col("fastestLapSpeed").cast("float"),
                                      "avgPitstop": col("avgPitstop").cast("float"),
                                      "grid": col("grid").cast("int"),
                                      "rank":col("rank").cast("int"),
                                     })

#Converting fastest lap time to miliseconds 
df_race_results = df_race_results.withColumn("fastestLapTime_ms",
    when(col("fastestLapTime") != "\\N",
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 1).cast("int") * 60000) +  # minutes
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 2).cast("int") * 1000) +  # seconds
        (regexp_extract(col("fastestLapTime"), r"(\d+):(\d+)\.(\d+)", 3).cast("int") * 1)      # milliseconds
    ).otherwise(None)
)
display(df_race_results)

In [0]:
df_pandas = df_race_results.toPandas()
X = df_pandas[['laps', 'fastestLap', 'fastestLapSpeed', 'avgPitstop', 'grid', 'rank', 'fastestLapTime_ms']]
y = df_pandas['positionOrder']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
